# 01 - Data Download
This notebook will be used to download the various datasets needed for the project. That includes:

| Dataset | Use | Code Complete? | Downloaded? |
| --- | --- | --- | --- |
| USGS/other federal boundary holder | site boundary that's used to select downloaded data | no | no |
| USGS DEM | digital elevation model used for SWE training | no | no |
| SNOTEL | timeseries used for SWE training | no | no |
| PRISM climate data | gridded reanalysis data used for SWE training | no | no |
| MACAv2 climate data | gridded future data used for SWE prediction | no | no |

## Step 1: Import Libraries

In [3]:
# Library management
import os
import pathlib

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # for SNOTEL API access
import zipfile

# Data Management
import pandas as pd
import xarray as xr

# Geospatial Management
import geopandas as gpd
from pygeohydro import WBD # site boundary based on watersheds
#import rasterio
import rioxarray as rxr

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
from shapely.geometry import box

In [ ]:
# You may need to install pygeohydro 

# if so, uncomment and run this block to install the 
# package to your current kernel

#%pip install pygeohydro

# For windows users - also run this

#%pip uninstall -y aiodns

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------------ --------------------- 1.8/4.0 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 8.6 MB/s  0:00:00

   -- -------------------------------------  1/16 [lxml]
   -- -------------------------------------  1/16 [lxml]
   -- -------------------------------------  1/16 [lxml]
   ------- --------------------------------  3/16 [h5netcdf]
   ------- --------------------------------  3/16 [h5netcdf]
   ------------ ---------------------------  5/16 [aiofiles]
   --------------- ------------------------  6/16 [pycares]
   ----------------- ----------------------  7/16 [owslib]
   ----------------- ----------------------  7/16 [owslib]
   ----------------- ----------------------  7/16 [owslib]
   ----------------- ----------------------  7/16 [owslib]
   -------------------- -------------------  8/16 [aio

## Step 2: Site Selection
In this step, I download the HUC6 watershed boundary for the Missouri Headwaters region to use as the bounding box for the project.

I've selected HUC6 watershed boundaries that will capture the following ranges:

**FILL IN**

**REASONING FOR HUC6 boundaries**

documentation on pygeohydro: https://docs.hyriver.io/readme/pygeohydro.html

Citation:
@article{Chegini_2021,
    author = {Chegini, Taher and Li, Hong-Yi and Leung, L. Ruby},
    doi = {10.21105/joss.03175},
    journal = {Journal of Open Source Software},
    month = {10},
    number = {66},
    pages = {1--3},
    title = {{HyRiver: Hydroclimate Data Retriever}},
    volume = {6},
    year = {2021}
}

### 2a: Download

In [4]:
# get HUC8 boundaries in SW Montana bounding box

# set bounding box
bbox = (-114.5, 44.3, -109.5, 47.0) # this is bigger than the ultimate study area

# get all HUC8 boundaries w/ WBD package
wbd = WBD("huc8")

# filter to bounding box
huc8_bbox = wbd.bygeom(box(*bbox), geo_crs=4326)

# Check total HUC8s in box
print(f"Total huc8s in bounding box: {len(huc8_bbox)}")
print(huc8_bbox[["huc8", "name"]].to_string())


Total huc8s in bounding box: 44
        huc8                          name
0   10030102       Upper Missouri-Dearborn
1   10070003                       Shields
2   10020006                       Boulder
3   10030103                         Smith
4   10030105                          Belt
5   10040103                        Judith
6   10040201             Upper Musselshell
7   17040202                  Upper Henrys
8   17040214                  Beaver-Camas
9   17040215                Medicine Lodge
10  17040216                         Birch
11  17040217                   Little Lost
12  17060201                  Upper Salmon
13  17060202                    Pahsimeroi
14  17060203         Middle Salmon-Panther
15  17060204                         Lemhi
16  17060205      Upper Middle Fork Salmon
17  17060206      Lower Middle Fork Salmon
18  17060207     Middle Salmon-Chamberlain
19  17060301                  Upper Selway
20  17060302                  Lower Selway
21  17060303          

In [6]:
# Filter to Missouri Headwaters headwaters (huc6 = 100200)
huc8_missouri = huc8_bbox[huc8_bbox["huc8"].str.startswith("1002")].copy()

# Check HUC8s in Missouri Headwaters
print(f"\nMissouri Headwaters HUC8s: {len(huc8_missouri)}")
print(huc8_missouri[["huc8", "name"]].to_string())



Missouri Headwaters HUC8s: 8
        huc8        name
2   10020006     Boulder
29  10020001    Red Rock
30  10020002  Beaverhead
31  10020003        Ruby
32  10020004    Big Hole
33  10020005   Jefferson
38  10020008    Gallatin
40  10020007     Madison


In [7]:
# Visual Inspection

# Set CRS
huc8_missouri = huc8_missouri.set_crs("EPSG:4326")

# Make plot
huc8_plot = huc8_missouri.hvplot.polygons(
    geo=True,
    tiles="EsriNatGeo",
    alpha=0.4,
    line_color="black",
    line_width=0.8,
    color="name",
    cmap="tab20",
    hover_cols=["huc8", "name"],
    title="Missouri Headwaters HUC8 Watersheds — SW Montana",
    width=900,
    height=700,
    legend=False
)

# Show plot
huc8_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]   (name,huc8)

This is currently missing SNOTEL sites in the Bridger range, and potentially in the Gallatins. Further, there might be some SNOTEL sites in the western Absarokas that would be worth including in the training dataset. Worth looking into adding more HUC8 areas. The western and southern boundaries look good (that's the continental divide); the northern boundary could catch the rest of the mountains to Helena but that might be too far of a reach.

https://ftpgeoinfo.msl.mt.gov/Data/Spatial/NonMSDI/NRCS/HUCEnv/MTHydroMap.pdf great visualization of all the HUC8s. Maybe add Shields, Upper Yellowstone, Yellowstone Headwaters

In [9]:
print(huc8_bbox[["huc8", "name"]].to_string())

        huc8                          name
0   10030102       Upper Missouri-Dearborn
1   10070003                       Shields
2   10020006                       Boulder
3   10030103                         Smith
4   10030105                          Belt
5   10040103                        Judith
6   10040201             Upper Musselshell
7   17040202                  Upper Henrys
8   17040214                  Beaver-Camas
9   17040215                Medicine Lodge
10  17040216                         Birch
11  17040217                   Little Lost
12  17060201                  Upper Salmon
13  17060202                    Pahsimeroi
14  17060203         Middle Salmon-Panther
15  17060204                         Lemhi
16  17060205      Upper Middle Fork Salmon
17  17060206      Lower Middle Fork Salmon
18  17060207     Middle Salmon-Chamberlain
19  17060301                  Upper Selway
20  17060302                  Lower Selway
21  17060303                        Lochsa
22  1701020

I'll do a basic search on the [SNOTEL API Demo](https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html#/Station%20Metadata/getStations) to see how many SNOTEL sites are in the three watersheds I'll consider adding:
- Shields (10070003): 4 SNOTEL sites (2 in Bridgers, 2 in Crazy Mountains)
- Upper Yellowstone (10070002): 5 SNOTEL sites, goes fairly far east
- Yellowstone Headwaters (10070001): 1 SNOTEL site in MT, 4 in WY

So, might be some important sites to capture here, or it could really spread out the study area. I might need to plot these SNOTEL sites on a map to see where they land.

Using the [MT State Library ArcGIS viewer](https://www.arcgis.com/apps/mapviewer/index.html?layers=1b0d0b21de9747d0b6a16e3428ae1eb6), I've found a few SNOTEL sites that are within 3 miles of the Missouri Headwaters watershed boundary:
- Brackett Creek (Bridgers)
- Sacajawea (Bridgers)
- Canyon (Yellowstone) 384:WY:SNTL
- WHite Elephant (Henry's Fork)
- Saddle Mtn (Lower East Fork Bitterroot)
- Moose Creek (Salmon)
- Barker Lakes (Upper Clark Fork)
- Frohner Meadow (Upper Missouri)

Based on these extra sites, I can finalize study area design. A 5km buffer around the Missouri Headwaters Basin will grab a few extra SNOTEL sites that seem important for the training data. However, a few lie on the other side of the Continental Divide, so I'll clip those out after filtering the sites. I can proceed with the study area boundary generation below using a spatial join and don't need to clumsily tack on extra HUCs as I thought. Then, in the SNOTEL processing step, I will grab metadata for all MT sites and filter to the study area. The table below - inspired by a conversation with Claude - cleanly explains the boundaries I'm using:

| Layer | Definition | Purpose |
| --- | --- | --- |
| Study area | HUC6 100200 (Missouri Headwaters Basin) dissolved | Spatial study definition, result maps |
| Buffer area | Study area + 5km buffer | SNOTEL, PRISM, MACA data selection |
| Continental Divide Clip | Buffer area clipped to remove buffer on the west side of the CD | SNOTEL, PRISM, MACA data selection |

### 2b: Spatial Join
need to join and dissolve

In [10]:
# Set study area boundary
mhw_gdf = huc8_missouri.dissolve().reset_index()[['geometry']]
mhw_gdf = mhw_gdf.set_crs('EPSG:4326')

# check it out
mhw_gdf

,geometry
0,"POLYGON ((-111.51646 44.64282, -111.51652 44.6..."


### 2c: Plot Boundary

In [43]:
# make the plot
study_area_plot = mhw_gdf.hvplot(
    geo=True,
    tiles="EsriNatGeo",
    alpha=0.4,
    line_color="black",
    line_width=0.8,
    title="Missouri Headwaters Basin (HUC6) Study Area",
    width=900,
    height=700,
    legend=False
)

# Show the plot
study_area_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

## Step 3: Access SNOTEL Data

- API Documentation: https://wcc.sc.egov.usda.gov/awdbRestApi/v3/api-docs
- Interactive API demo: https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html
- GitHub Demo repo: https://github.com/nrcs-nwcc/iow_awdb_rest_api_demo

### 3a: Download station metadata

This step gets the metadata for each station, which I can then filter using the watershed boundary.

In [88]:
### From Claude:
# this downloads station metadata for all the stations in MT

# Get station metadata for Montana
url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"
params = {
    "stationTriplets": "*:MT:SNTL", # all SNOTEL stations in Montana
    "elements": ["PREC", "WTEQ", "TMIN", "TMAX"]    # filter for stations measuring SWE, precip, tmin, and tmax
}
# Query server
response = requests.get(url, params=params)
# Get the station list
mt_stations = response.json()

In [89]:
# Inspect
print(f"There are {len(mt_stations)} SNOTEL stations in Montana.")

# Check out the formatting
mt_stations[:3]

There are 96 SNOTEL stations in Montana.


[{'stationTriplet': '916:MT:SNTL',
  'stationId': '916',
  'stateCode': 'MT',
  'networkCode': 'SNTL',
  'name': 'Albro Lake',
  'dcoCode': 'MT',
  'countyName': 'Madison',
  'huc': '100200050701',
  'elevation': 8500.0,
  'latitude': 45.59723,
  'longitude': -111.95902,
  'dataTimeZone': -8.0,
  'shefId': 'ABRM8',
  'operator': 'NRCS',
  'beginDate': '1996-09-01 00:00',
  'endDate': '2100-01-01 00:00',
  'associatedHucs': ['100200050601', '100200050702', '100200071101']},
 {'stationTriplet': '307:MT:SNTL',
  'stationId': '307',
  'stateCode': 'MT',
  'networkCode': 'SNTL',
  'name': 'Badger Pass',
  'dcoCode': 'MT',
  'countyName': 'Pondera',
  'huc': '100302010201',
  'elevation': 6870.0,
  'latitude': 48.13091,
  'longitude': -113.02311,
  'dataTimeZone': -8.0,
  'shefId': 'BADM8',
  'operator': 'NRCS',
  'beginDate': '1968-10-01 00:00',
  'endDate': '2100-01-01 00:00',
  'associatedHucs': ['100302010202',
   '100302010204',
   '100302010601',
   '100302010603',
   '170102070101',
 

[The USDA](https://www.nrcs.usda.gov/state-offices/montana/montana-snow-survey/frequently-asked-snow-survey-questions-montana) confirms the number of stations. Success! There is one station in Wyoming that needs to be added though - I'll download that station and append it.

In [90]:
# Get station metadata for Montana
url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"
wy_params = {
    "stationTriplets": "384:WY:SNTL", # all SNOTEL stations in Montana
    "elements": ["PREC", "WTEQ", "TMIN", "TMAX"]    # filter for stations measuring SWE, precip, tmin, and tmax
}
# Query server
response = requests.get(url, params=wy_params)
# Get the station list
wy_station = response.json()

# Inspect
wy_station

[{'stationTriplet': '384:WY:SNTL',
  'stationId': '384',
  'stateCode': 'WY',
  'networkCode': 'SNTL',
  'name': 'Canyon',
  'dcoCode': 'MT',
  'countyName': 'Park',
  'huc': '100700010705',
  'elevation': 7870.0,
  'latitude': 44.71961,
  'longitude': -110.51084,
  'dataTimeZone': -8.0,
  'shefId': 'CANW4',
  'operator': 'NRCS',
  'beginDate': '1960-10-01 00:00',
  'endDate': '2100-01-01 00:00',
  'associatedHucs': ['100200070105',
   '100700010701',
   '100700010702',
   '100700010703',
   '100700010706']}]

In [96]:
# Append WY Station to station list via concatenation
stations = mt_stations + wy_station
len(stations)
type(stations)

list

In [97]:
# Plot the stations

# Convert list to gdf
# make it a df first
station_df = pd.DataFrame(stations)
# then convert to GDF
sntl_station_gdf = gpd.GeoDataFrame(
    station_df,
    geometry=gpd.points_from_xy(station_df['longitude'], station_df['latitude']),
    crs = 'EPSG:4326'
    )

sntl_station_gdf

,stationTriplet,stationId,stateCode,networkCode,name,dcoCode,countyName,huc,elevation,latitude,longitude,dataTimeZone,shefId,operator,beginDate,endDate,associatedHucs,pedonCode,geometry
0,916:MT:SNTL,916,MT,SNTL,Albro Lake,MT,Madison,100200050701,8500.0,45.59723,-111.95902,-8.0,ABRM8,NRCS,1996-09-01 00:00,2100-01-01 00:00,"[100200050601, 100200050702, 100200071101]",NaN,POINT (-111.95902 45.59723)
1,307:MT:SNTL,307,MT,SNTL,Badger Pass,MT,Pondera,100302010201,6870.0,48.13091,-113.02311,-8.0,BADM8,NRCS,1968-10-01 00:00,2100-01-01 00:00,"[100302010202, 100302010204, 100302010601, 100...",NaN,POINT (-113.02311 48.13091)
2,311:MT:SNTL,311,MT,SNTL,Banfield Mountain,MT,Lincoln,170101011202,5580.0,48.57120,-115.44573,-8.0,BANM8,NRCS,1968-10-01 00:00,2100-01-01 00:00,"[170101011001, 170101011002, 170101011106, 170...",NaN,POINT (-115.44573 48.5712)
3,313:MT:SNTL,313,MT,SNTL,Barker Lakes,MT,Deer Lodge,170102010304,8250.0,46.09713,-113.13038,-8.0,BRLM8,NRCS,1977-08-01 00:00,2100-01-01 00:00,"[100200040703, 170102010208, 170102010301]",NaN,POINT (-113.13038 46.09713)
4,315:MT:SNTL,315,MT,SNTL,Basin Creek,MT,Silver Bow,170102010201,7120.0,45.79737,-112.52047,-8.0,BSCM8,NRCS,1976-10-01 00:00,2100-01-01 00:00,"[100200041201, 100200041203, 100200050501, 170...",NaN,POINT (-112.52047 45.79737)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,924:MT:SNTL,924,MT,SNTL,West Yellowstone,MT,Gallatin,100200070205,6680.0,44.65866,-111.09199,-8.0,WYSM8,NRCS,1966-10-01 00:00,2100-01-01 00:00,"[100200070203, 100200070204]",NaN,POINT (-111.09199 44.65866)
93,858:MT:SNTL,858,MT,SNTL,Whiskey Creek,MT,Gallatin,100200070203,6790.0,44.61088,-111.14998,-8.0,WSKM8,NRCS,1967-10-01 00:00,2100-01-01 00:00,"[100200070202, 100200070204, 100200070205, 170...",NaN,POINT (-111.14998 44.61088)
94,862:MT:SNTL,862,MT,SNTL,White Mill,MT,Park,100700060202,8730.0,45.04575,-109.90987,-8.0,WHTM8,NRCS,1966-10-01 00:00,2100-01-01 00:00,"[100700010602, 100700050101]",39521,POINT (-109.90987 45.04575)
95,876:MT:SNTL,876,MT,SNTL,Wood Creek,MT,Lewis and Clark,100301040202,5970.0,47.44847,-112.81428,-8.0,WODM8,NRCS,1978-10-01 00:00,2100-01-01 00:00,"[100301040201, 100301040203, 100301040302, 100...",NaN,POINT (-112.81428 47.44847)


In [98]:
# Plot the study area and the stations
sntl_mt_plot = (
    # Plot study area polygon
    mhw_gdf.hvplot.polygons(
        geo=True,
        tiles="EsriNatGeo",
        fill_color=None,  # Clear inside
        line_color="black",
        line_width=1.5,
        title="SNOTEL Stations in Missouri Headwaters Basin",
        width=900,
        height=700,
    )
    # SNOTEL Stations
    * sntl_station_gdf.hvplot.points(
        geo=True, 
        color="blue", 
        alpha=1,
        hover_cols = ['stationTriplet', 'name', 'countyName', 'huc'],
        size=40, 
        legend=False
    )
)

# Show plot
sntl_mt_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]
   .Points.I   :Points   [Longitude,Latitude]   (stationTriplet,name,countyName,huc)

The plot above shows the SNOTEL sites in MT (and one in WY that will be included). You can explore the map and see the names of stations, and get a sense of which ones fit into the study area or are very close.

### 3b: Filter stations

#### 3c: Test download func for one site

#### 3d: Download all sites

### 3?: SNOTEL Site Map
Make a map of SNOTEL sites to (a) confirm site locations and (b) provide a plot of where sites are located.

## Step 4: Access DEM

## Step 5: Access PRISM Data

## Step 6: Access MACAv2 Data

### 6a: Data dir

In [ ]:
# make directory for climate data

clim_dir = os.path.join(data_dir, 'model_data')
os.makedirs(clim_dir, exist_ok = True)

### 6b: Define functions

Functions to help process climate data.

In [ ]:
# convert temp from Kelvin to F function
def convert_K_to_F(da):
    '''
    Converts an xarray DataArray from Kelvin to Fahrenheit 

    Args:
    da (DataArray):
        DA of temperature values
    '''

    # Perform conversion pixel-wise
    da_f = (da * 1.8) - 459.67
    
    return da_f

In [ ]:
# convert longitude
def convert_longitude(longitude):

    '''
    This function cleans up datasets read from NetCDF files 
    in order to work with the other datasets used in this notebook.
    It converts lon from 0 to 360 to -180 to 180 

    Args:
    =====
    longitude (DataArray):
        DA from larger Dataset that contains longitude data

    Returns:
    ========
    longitude (DataArray)
    '''

    return longitude - 360 if longitude > 180 else longitude 

In [ ]:
# Function to download model data and process it
# we will need to update this for the summer class - currently this function takes an annual mean.

def download_and_process_maca_data(
        site_gdf, site_name,
        models, rcp_value,
        variable, year_range,
        maca_dir):
    '''
    Downloads and processes MACA data for a given site.
    Processing includes clipping bounds and rewriting longitude values
    
    Args:
    =====
    site_gdf (geopandas.GeoDataFrame):
        Site boundary, used for clipping
    site_name (str):
        Name of site
    models (list):
        List of models to programmatically download
    rcp_value (str):
        RCP value for MACA url
    variable (str):
        Variable of interest
    year_range (str):
        Range of years to be downloaded
    maca_dir (str):
        Path for data to be saved
    '''

    # Initialize results list
    results = []

    # count total steps for progress bar
    total_steps = len(models) * len(year_range)

    # download and process files
    with tqdm(total=total_steps, desc=f"Processing {site_name}") as pbar:
        # loop through models
        for model in models:

            # hold onto each time chunk for aggregation at the end
            model_chunks = []
            # loop through year chunks
            for years in year_range:

                # Programmatically determine file path and download url
                file_name = f'maca_{model}_{site_name}_{rcp_value}_{years}_CONUS_monthly.nc'
                maca_path = os.path.join(maca_dir, file_name)
                maca_url = (
                    f'http://thredds.northwestknowledge.net:8080/thredds/dodsC/MACAV2/{model}/'
                    f'macav2metdata_{variable}_{model}_r1i1p1_{rcp_value}_{years}_CONUS_monthly.nc'
                )
                

                # Download or open file
                if not os.path.exists(maca_path):
                    print(f"{model}, {years}, {rcp_value} downloaded successfully!")
                    maca_ds = xr.open_dataset(maca_url, mask_and_scale = True).squeeze()
                    maca_ds.to_netcdf(maca_path)
                else:
                    print(f"{model}, {years}, {rcp_value} already exists")
                    maca_ds = xr.open_dataset(maca_path, mask_and_scale = True).squeeze()

                # Convert longitude
                # reassign lon
                maca_ds = maca_ds.assign_coords(
                    lon = ('lon', [convert_longitude(l) for l in maca_ds.lon.values])
                )

                # set crs
                maca_ds = maca_ds.rio.write_crs('EPSG:4326')
                
                # set spatial dimension to define da as spatial
                maca_ds = maca_ds.rio.set_spatial_dims(
                    x_dim = 'lon',
                    y_dim = 'lat'
                )

                # set site bounds
                site_rpj = site_gdf.to_crs(maca_ds.rio.crs)
                site_bounds = site_rpj.total_bounds

                # write no data as NaN
                maca_ds['air_temperature'].rio.write_nodata(np.nan, inplace=True)

                # crop to bounding box
                maca_ds_crop = maca_ds.rio.clip_box(*site_bounds)
                
                # Append this specific 5-year chunk to the model list
                model_chunks.append(maca_ds_crop)

                # Update progress bar
                pbar.update(1)

            # Combine all year chunks for this model into one Dataset
            combined_model_ds = xr.concat(model_chunks, dim='time')
            
            # Convert temperature
            combined_model_ds['air_temperature'] = convert_K_to_F(combined_model_ds['air_temperature'])

            # Mask No Data values for mean
            valid_data = combined_model_ds.where(combined_model_ds > -100)

            # Calculate the mean across the time dimension
            mean_model_ds = valid_data.mean(dim='time')

            # convert to DataArray
            mean_model_da = mean_model_ds['air_temperature']

            # rewrite NaN values
            mean_model_da.rio.write_nodata(np.nan, inplace=True)
            
            # store data
            results.append({
                'site_name': site_name,
                'model': model,
                'rcp_value': rcp_value,
                'variable': variable,
            #    'date_range': years,
                'data': mean_model_da
            })
            
    return results

### 6c: Model Selection
Do I need this section? TBD

1. Get total bounds for each site to input into MACA [scatterplot viewer](https://climate.northwestknowledge.net/MACA/vis_scatterplot.php). 
2. Visualize ensemble spread. We used the mid-century time period, and plotted summer temperature change against winter precipitation change.
3. Find models that capture 4 scenarios: Hot-Wet, Hot-Dry, Cold-Wet, Cold-Dry. We selected models from the RCP4.5 pathway.

### 6d: Download Model Data

In [ ]:
# define year range and model lists

# define year chunks
### REVISE THIS
historic_range = ['1970_1974', '1975_1979', '1980_1984', 
                  '1985_1989', '1990_1994', '1995_1999']
future_range = ['2036_2040', '2041_2045', '2046_2050', 
                '2051_2055', '2056_2060', '2061_2065']

# define models
models = {
    "site_name": [
        "model_name1", "model_name2", "etc"],
}

In [ ]:
# set site names and gdfs
sites = {
    # fill in
}

# set scenarios
scenarios = [
    {'name': 'historical', 'range': historic_range},
    {'name': 'rcp45', 'range': future_range}
]

# initialize list of all the results
model_results = {}

# loop through sites and scenarios
for site_name, gdf in sites.items():
    for scn in scenarios:
        # Key for results
        result_key = f"{site_name.lower()}_{scn['name']}"

        # Print status update
        print(f"\n--- Running {site_name} | {scn['name']} ---")

        model_results[result_key] = download_and_process_maca_data(
            site_gdf = gdf,
            site_name = site_name,
            models = models[site_name.lower()],
            rcp_value = scn['name'],
            # downloading max temp here, but can change to other variables
            variable = 'tasmax',
            year_range = scn['range'],
            maca_dir = clim_dir
        )

### 6e: Save Model Data

THIS SECTION NEEDS TO BE COMPLETED.